### Compactación mediante Z-ORDER
Es una técnica de organización física de los datos que busca colocar filas con valores similares cerca unas de otras dentro de los archivos de una tabla.
> 1. Crear **500,000,000** de registros aleatorios
> 2. Guardar los registros en una tabla llamada "**demo.delta_lake.order_data**"
> 3. Consultar la tabla sin compactación "**OPTIMIZE**" y "**Z-ORDER**"
> 4. Ejecutar "**OPTIMIZE**" y "**Z-ORDER (city, order_date)**"
> 5. Consultar la tabla con compactación "**OPTIMIZE**" y "**Z-ORDER**"

#### 1. Crear **500,000,000** de registros aleatorios

In [0]:
%python
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

# Número de registros
num_records = 500000000

# Paso 1 - Crear un Dataframe con 50,000 registros
df = spark.range(0, num_records)

# Paso 2 - Agregar las columnas requeridas
df = (
    df.withColumn("order_id", F.concat(F.lit("ODR_"), F.lpad(F.col("id"), 5, "0")))
      .withColumn("city", F.when(F.rand() < 0.4, "New York")
                           .when(F.rand() < 0.7, "San Diego")
                           .otherwise("Dallas"))
      .withColumn("order_date",
                  F.date_add(F.lit("2026-01-10"), (F.rand() * 5).cast("int")))
      .withColumn("weight", F.round((F.rand() * 10).cast("double"), 4))
      .drop("id")
)

df.show(10)
print("Total records:", df.count())

#### 2. Guardar los registros en una tabla llamada "**demo.delta_lake.order_data**"

In [0]:
%python
df.writeTo("demo.delta_lake.order_data").createOrReplace()

#### 3. Consultar la tabla sin compactación "**OPTIMIZE**" y "**Z-ORDER**"

In [0]:
SELECT * FROM demo.delta_lake.order_data
WHERE city = 'New York' AND order_date = '2026-01-10';

#### 4. Ejecutar "**OPTIMIZE**" y "**Z-ORDER (city, order_date)**"

In [0]:
OPTIMIZE demo.delta_lake.order_data
ZORDER BY (city, order_date);

#### 5. Consultar la tabla con compactación "**OPTIMIZE**" y "**Z-ORDER**"

In [0]:
SELECT * FROM demo.delta_lake.order_data
WHERE city = 'New York' AND order_date = '2026-01-10';